# 01 — Cleaning & ops marts

Landing extracts → typed cleaned tables → analytics marts for the QuickCommerce ops tower.

Partial deliveries still count toward GMV / SLA (customer received something).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path('..')
LANDING, CLEANED, MARTS = ROOT / 'data' / 'landing', ROOT / 'data' / 'cleaned', ROOT / 'data' / 'marts'
REPORTS = ROOT / 'reports'
CLEANED.mkdir(parents=True, exist_ok=True)
MARTS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

stores = pd.read_csv(LANDING / 'dim_stores_raw.csv')
skus = pd.read_csv(LANDING / 'dim_skus_raw.csv')
riders = pd.read_csv(LANDING / 'dim_riders_raw.csv')
orders = pd.read_csv(LANDING / 'orders_raw.csv')
lines = pd.read_csv(LANDING / 'order_lines_raw.csv')
stockouts = pd.read_csv(LANDING / 'stockout_events_raw.csv')
print(f'Landing: orders={len(orders):,} lines={len(lines):,}')


In [ ]:
o = orders.copy()
o['ordered_at'] = pd.to_datetime(o['ordered_at'])
o['order_date'] = pd.to_datetime(o['order_date'])
o['is_delivered'] = o['status'].isin(['Delivered', 'Partial']).astype(int)
o['is_cancelled'] = (o['status'] == 'Cancelled').astype(int)
assert (o['gmv'] >= 0).all()
assert o['order_id'].is_unique

stores.to_csv(CLEANED / 'dim_stores.csv', index=False)
skus.to_csv(CLEANED / 'dim_skus.csv', index=False)
riders.to_csv(CLEANED / 'dim_riders.csv', index=False)
o.to_csv(CLEANED / 'fact_orders.csv', index=False)
lines.to_csv(CLEANED / 'fact_order_lines.csv', index=False)
stockouts.to_csv(CLEANED / 'fact_stockouts.csv', index=False)

stores.to_csv(MARTS / 'dim_store.csv', index=False)
skus.to_csv(MARTS / 'dim_sku.csv', index=False)
riders.to_csv(MARTS / 'dim_rider.csv', index=False)

dates = pd.DataFrame({'order_date': pd.date_range(o['order_date'].min(), o['order_date'].max())})
dates['date_key'] = dates['order_date'].dt.strftime('%Y%m%d').astype(int)
dates['dow'] = dates['order_date'].dt.day_name()
dates['is_weekend'] = dates['order_date'].dt.weekday.ge(5).astype(int)
dates['week_num'] = dates['order_date'].dt.isocalendar().week.astype(int)
dates.to_csv(MARTS / 'dim_date.csv', index=False)

fact = o.copy()
fact['date_key'] = fact['order_date'].dt.strftime('%Y%m%d').astype(int)
fact['gross_margin'] = fact['net_gmv'] - fact['cogs']
fact.to_csv(MARTS / 'fact_orders.csv', index=False)
lines.to_csv(MARTS / 'fact_order_lines.csv', index=False)
stockouts.to_csv(MARTS / 'fact_stockouts.csv', index=False)
print('cleaned + base marts written')
fact[['gmv','net_gmv','is_delivered','is_cancelled']].describe()


In [ ]:
# Rebuild key KPI marts if helper columns exist
delivered = fact[fact['is_delivered'] == 1]
mart_daily = fact.groupby('order_date', as_index=False).agg(
    orders=('order_id', 'count'),
    delivered_orders=('is_delivered', 'sum'),
    cancelled_orders=('is_cancelled', 'sum'),
    gmv=('gmv', 'sum'),
    net_gmv=('net_gmv', 'sum'),
)
if 'sla_hit' in fact.columns:
    sla = delivered.groupby('order_date', as_index=False).agg(sla_hit_pct=('sla_hit', 'mean'), o2d_p50=('o2d_min', 'median') if 'o2d_min' in fact.columns else ('sla_hit', 'mean'))
    mart_daily = mart_daily.merge(sla, on='order_date', how='left')
    if 'sla_hit_pct' in mart_daily.columns:
        mart_daily['sla_hit_pct'] = (mart_daily['sla_hit_pct'] * 100).round(2)
mart_daily.to_csv(MARTS / 'mart_daily.csv', index=False)

kpi = {
    'orders': int(len(fact)),
    'delivered': int(fact['is_delivered'].sum()),
    'gmv': round(float(fact['gmv'].sum()), 2),
    'net_gmv': round(float(fact['net_gmv'].sum()), 2),
}
if 'sla_hit' in fact.columns:
    kpi['sla_hit_pct'] = round(float(delivered['sla_hit'].mean()) * 100, 2)
(REPORTS / 'kpi_summary.json').write_text(json.dumps(kpi, indent=2))
print(kpi)
mart_daily.tail()
